___
# <center>Regressão logística no scikit-learn, validação cruzada e o ponto de corte</center>
___

## Aula 23

Nas aulas 20 e 21, a regressão logística serviu para **explicar**: olhamos os
coeficientes, os intervalos e os valores-p. Hoje o objetivo muda para
**prever**: queremos um modelo que acerte em processos que ele nunca viu. Por
isso o jeito de avaliar também muda. O que importa não é o ajuste aos dados que
o modelo já conhece, e sim o desempenho em casos novos.

O caminho do notebook é o mesmo dos slides:

1. separar **treino e teste**, e guardar o teste;
2. **preparar** as colunas para o modelo e ajustar a regressão logística;
3. medir o desempenho em casos novos com a **validação cruzada**;
4. escolher o **`C`**, o hiperparâmetro de complexidade, pela validação cruzada;
5. abrir o teste uma vez e **escolher o ponto de corte** pela necessidade de
   quem vai usar o modelo.

O notebook continua na aula 24, e os dois juntos são o ponto de partida do
Projeto 08.

___
<div id="indice"></div>

## Índice

- [A pergunta e a base](#base)
- [1. Treino, teste e o modelo trivial](#parte1)
- [2. O preparo das colunas](#parte2)
- [3. A regressão logística no scikit-learn](#parte3)
- [4. Validação cruzada](#parte4)
- [5. Escolher o `C`](#parte5)
- [6. O modelo final, o teste e o ponto de corte](#parte6)

___
<div id="base"></div>

# A pergunta e a base

Usamos a base do **Projeto 08**: acórdãos reais de recursos criminais do TJSP,
de 2016 a 2023, com uma linha por acórdão. A pergunta de hoje, porém, não é a do
projeto. Queremos prever se a decisão terá **voto divergente**, ou seja, se será
por maioria.

**Por que isso interessa.** No processo penal, uma decisão não unânime
desfavorável ao réu abre os **embargos infringentes** (art. 609, parágrafo único,
do CPP). Uma defensoria com muitos recursos não consegue acompanhar todos de
perto. Um modelo que aponte os casos com mais chance de divergência ajuda a
decidir onde vale a pena fazer sustentação oral ou entregar memoriais.

**Os pacotes.** A primeira célula carrega o que vamos usar. Leia os comentários:
cada linha do scikit-learn é uma peça que aparece mais adiante.

In [ ]:
# pandas: tabelas (a base, os resultados)
import pandas as pd
# plotnine: gráficos, com a mesma gramática do ggplot
from plotnine import *

# scikit-learn (sklearn): o pacote de machine learning. Cada import é uma peça:
from sklearn.model_selection import train_test_split   # separa treino e teste
from sklearn.model_selection import StratifiedKFold     # divide o treino em partes
from sklearn.model_selection import cross_val_score     # a validação cruzada
from sklearn.model_selection import cross_val_predict   # probabilidades de validação
from sklearn.compose import make_column_transformer     # um preparo para cada tipo de coluna
from sklearn.pipeline import make_pipeline              # encadeia etapas num objeto só
from sklearn.impute import SimpleImputer                # preenche valores faltantes
from sklearn.preprocessing import OneHotEncoder         # categorias viram colunas 0/1
from sklearn.preprocessing import StandardScaler        # números ficam na mesma escala
from sklearn.linear_model import LogisticRegression     # a regressão logística
from sklearn.metrics import roc_auc_score               # a AUC

A base tem estas colunas:

| coluna | o que é |
|---|---|
| `assunto` | assunto do recurso (roubo majorado, furto qualificado etc.) |
| `camara` | câmara criminal que julgou |
| `comarca` | comarca de origem |
| `polo_mp` | o Ministério Público recorreu (`Ativo`) ou é recorrido (`Passivo`) |
| `ano_publicacao` | ano em que o acórdão foi publicado |
| `tempo` | anos entre a distribuição e o julgamento |
| `rel_idade`, `rel_faculdade`, `rel_tempo_magistratura`, `rel_tipo`, `rel_quinto` | características de quem relatou |
| `negou` | o recurso foi integralmente negado (a resposta do Projeto 08) |
| `unanimidade` | a decisão foi unânime ou por maioria |

A célula abaixo lê a base direto do site da disciplina e cria a resposta,
`divergiu`, que vale 1 quando a decisão foi por maioria.

In [ ]:
base = pd.read_csv("https://jtrecenti.github.io/cdad2-202662/ex08-competicao/dados/treino.csv")
base = base.dropna(subset=["unanimidade"])                        # 39 acórdãos sem a informação
base["divergiu"] = (base["unanimidade"] == "Maioria").astype(int)  # 1 se houve voto divergente
print(base.shape)
base.head(3)

Quantos acórdãos tiveram divergência?

In [ ]:
base["divergiu"].value_counts()

A divergência é **rara**: cerca de 2% dos acórdãos. Isso vai pesar na escolha da
medida de desempenho.

**Por que a coluna `negou` fica de fora.** O resultado do recurso sai no mesmo
julgamento que a divergência. Quem quer prever a divergência **antes** do
julgamento não conhece o resultado. Usar essa coluna seria **vazamento**: o
modelo pareceria ótimo no papel e seria inútil na prática. O tema volta na aula
24.

[Volta ao Índice](#indice)

___
<div id="parte1"></div>

# 1. Treino, teste e o modelo trivial

Antes de qualquer modelo, guardamos uma parte da base na **caixa do teste**:
25% dos acórdãos, sorteados, que só serão usados no fim, uma vez. É a única
forma honesta de saber como o modelo vai em casos novos.

Na linguagem do scikit-learn:

- `X` são as colunas usadas para prever (as variáveis explicativas);
- `y` é a resposta, `divergiu`;
- `train_test_split` sorteia quem vai para o treino e quem vai para o teste;
- `stratify` garante a mesma proporção de divergências nas duas partes. Com só
  2% de casos, um sorteio sem isso poderia deixar o teste quase sem divergências;
- `random_state` fixa o sorteio, para todo mundo ter o mesmo resultado.

As colunas que o modelo vai usar ficam em duas listas, conforme o tipo.

In [ ]:
categoricas = ["assunto", "camara", "comarca", "polo_mp", "rel_faculdade", "rel_tipo", "rel_quinto"]
numericas = ["ano_publicacao", "tempo", "rel_idade", "rel_tempo_magistratura"]
colunas = categoricas + numericas

Complete a proporção do teste: 25%.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    base[colunas], base["divergiu"], test_size=____, stratify=base["divergiu"], random_state=23)

print("treino:", len(X_treino), "acórdãos;", "teste:", len(X_teste), "acórdãos")
print("divergências no treino:", y_treino.sum(), "| no teste:", y_teste.sum())

**O modelo trivial.** Antes de comemorar qualquer acerto, vale perguntar: quanto
acerta um "modelo" que sempre diz que **não** haverá divergência?

**Pergunta 1.** Qual seria a acurácia desse modelo trivial? Por que a acurácia
não serve para comparar modelos neste problema? Que medida usamos no lugar, e
por quê?

[Volta ao Índice](#indice)

___
<div id="parte2"></div>

# 2. O preparo das colunas

A regressão logística só trabalha com **números**, e sem valores faltantes. A
base tem categorias (o assunto, a câmara, a comarca) e alguns faltantes (as
informações de quem relatou não existem para juízes substitutos). Antes de
ajustar, cada coluna precisa ser transformada:

| problema | o que o preparo faz |
|---|---|
| categoria faltante | vira a categoria `"não informado"` |
| categoria em texto | vira uma coluna 0/1 por categoria (as **dummies** da aula 20) |
| categoria rara (menos de 5 acórdãos) | junta-se às outras raras numa coluna só |
| número faltante | vira a mediana da coluna |
| números em escalas diferentes | ficam com média 0 e desvio 1 (padronização) |

A padronização importa por causa da penalidade da parte 5: ela pune
coeficientes grandes, e o tamanho de um coeficiente depende da escala da
variável. Na mesma escala, a penalidade trata todas igualmente.

A função `cria_preparo()` monta esse preparo. Leia os comentários e rode.

In [ ]:
categoricas = ["assunto", "camara", "comarca", "polo_mp", "rel_faculdade", "rel_tipo", "rel_quinto"]
numericas = ["ano_publicacao", "tempo", "rel_idade", "rel_tempo_magistratura"]


def cria_preparo():
    """Transforma as colunas da base em números que o modelo consegue usar."""
    # colunas categóricas: o faltante vira a categoria "não informado", e cada
    # categoria vira uma coluna 0/1 (categorias com menos de 5 acórdãos se juntam)
    categorias = make_pipeline(
        SimpleImputer(strategy="constant", fill_value="não informado"),
        OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False),
    )
    # colunas numéricas: o faltante vira a mediana, e todas ficam com média 0 e
    # desvio padrão 1, para que a penalidade trate todas do mesmo jeito
    numeros = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    preparo = make_column_transformer(
        (categorias, categoricas), (numeros, numericas), verbose_feature_names_out=False)
    return preparo.set_output(transform="pandas")   # devolve uma tabela com nomes

**Vamos ver o que o preparo faz.** O `fit_transform` aprende o preparo no treino
(quais categorias existem, as medianas, as médias e os desvios) e já devolve a
tabela transformada. Compare o tamanho antes e depois.

In [ ]:
preparo = cria_preparo()
tabela = preparo.fit_transform(X_treino)

print("antes:", X_treino.shape[1], "colunas")
print("depois:", tabela.shape[1], "colunas")
tabela.head(3)

As 11 colunas viraram mais de 250, quase todas de 0 e 1. Vejamos um acórdão só,
antes e depois. Para o depois, mostramos apenas as colunas que não são zero.

In [ ]:
um = X_treino.index[0]
print("ANTES")
print(X_treino.loc[um])
print()
print("DEPOIS (só as colunas diferentes de zero)")
depois = tabela.loc[um]
print(depois[depois != 0])

Cada categoria do acórdão virou um 1 na coluna correspondente, e os números
viraram distâncias até a média, em desvios padrão. Um `tempo` de -0,7 quer
dizer que esse recurso foi julgado mais rápido que a média.

**Pergunta 2.** Se o preparo fosse ajustado na base inteira (treino e teste
juntos), as médias e os desvios usados na padronização dependeriam também do
teste. Por que isso seria um problema, ainda que pequeno?

[Volta ao Índice](#indice)

___
<div id="parte3"></div>

# 3. A regressão logística no scikit-learn

No scikit-learn, todo modelo funciona do mesmo jeito:

- `fit(X, y)` **ajusta**: aprende os coeficientes a partir dos exemplos;
- `predict_proba(X)` **prevê**: devolve, para cada linha, a probabilidade de
  cada resposta. A coluna `[:, 1]` é a probabilidade de `1`, a divergência.

O `make_pipeline` junta o preparo e o modelo num objeto só. Assim, quando
chamamos `fit`, o preparo é aprendido e o modelo é ajustado na sequência, sempre
com os mesmos dados. Isso evita o problema da pergunta 2.

Começamos com `C=10000`, que quer dizer **quase nenhuma penalidade**: é o modelo
mais complexo possível, perto do que o statsmodels daria. (`max_iter` só dá ao
algoritmo mais passos para convergir.) Complete o nome do método que ajusta.

In [ ]:
quase_sem_penalidade = make_pipeline(cria_preparo(), LogisticRegression(C=10000, max_iter=3000))
quase_sem_penalidade.____(X_treino, y_treino)

prob_treino = quase_sem_penalidade.predict_proba(X_treino)[:, 1]
print("AUC no treino:", round(roc_auc_score(y_treino, prob_treino), 3))

**Pergunta 3.** A AUC no treino passou de 0,9. É uma boa notícia? Pense no
exercício 4 da aula 22 antes de seguir.

[Volta ao Índice](#indice)

___
<div id="parte4"></div>

# 4. Validação cruzada

Para saber como o modelo vai em casos novos **sem abrir o teste**, usamos a
validação cruzada, só dentro do treino:

1. o treino é dividido em 5 partes (`StratifiedKFold`, que mantém a proporção de
   divergências em cada parte);
2. em cada rodada, o pipeline inteiro (preparo e modelo) é ajustado em 4 partes
   e avaliado na quinta, que ele não viu;
3. cada parte é avaliação uma vez, e o resultado é a média das 5 rodadas.

O `cross_val_score` faz as 5 rodadas de uma vez. Com `scoring="roc_auc"`, a
medida de cada rodada é a AUC. Complete o número de partes.

In [ ]:
pastas = StratifiedKFold(____, shuffle=True, random_state=23)

auc = cross_val_score(quase_sem_penalidade, X_treino, y_treino, cv=pastas, scoring="roc_auc")
print("AUC de cada rodada:", auc.round(3))
print("média:", auc.mean().round(3))

**Por que a AUC na validação cruzada, e não a acurácia ou a sensibilidade?**
Acurácia, sensibilidade e precisão dependem de um **ponto de corte**, e o corte
ainda não foi escolhido: ele depende de quem vai usar o modelo (parte 6). Se a
comparação entre modelos dependesse do corte, um modelo poderia vencer num corte
e perder em outro. A AUC olha **todos os cortes de uma vez**: mede se o modelo
ordena bem os casos. Primeiro escolhemos o modelo que melhor ordena; depois, o
corte.

**Pergunta 4.** Compare a AUC no treino (parte 3) com a média da validação
cruzada. O que a diferença indica? As 5 rodadas deram valores parecidos?

[Volta ao Índice](#indice)

___
<div id="parte5"></div>

# 5. Escolher o `C`

O `C` é o **hiperparâmetro de complexidade** da regressão logística: quanto o
ajuste paga por coeficientes grandes. **`C` menor, penalidade maior**, e os
coeficientes encolhem para perto de zero. Com `C` grande demais, o modelo decora
(sobreajuste); com `C` pequeno demais, não aprende nada (subajuste).

O `C` não sai do `fit`: precisa ser escolhido **antes**. A forma correta é
testar vários valores e comparar a AUC na validação cruzada, sem olhar o teste.
O laço abaixo faz isso. Complete o valor que vai para o modelo.

In [ ]:
linhas = []
for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1, 10, 100, 10000]:
    modelo = make_pipeline(cria_preparo(), LogisticRegression(C=____, max_iter=3000))
    auc = cross_val_score(modelo, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean()
    linhas.append({"C": C, "auc_validacao": round(auc, 3)})

por_c = pd.DataFrame(linhas)
por_c

O gráfico ajuda a ver o desenho. O eixo do `C` está em escala logarítmica,
porque os valores vão de 0,003 a 10.000.

In [ ]:
(ggplot(por_c, aes(x="C", y="auc_validacao"))
 + geom_line(color="#E4002B", size=1)
 + geom_point(color="#E4002B", size=3)
 + scale_x_log10()
 + labs(x="C (mais penalidade para a esquerda)", y="AUC na validação cruzada")
 + theme_minimal())

**Pergunta 5.**

a) Qual `C` teve a maior AUC na validação cruzada?

b) O que acontece com a AUC quando a penalidade é muito fraca? E quando é forte
demais? Use as palavras **sobreajuste** e **subajuste**.

c) Os valores entre 0,3 e 1 ficaram muito parecidos. Faz sentido brigar pela
terceira casa decimal? O que você levaria em conta para escolher entre eles?

[Volta ao Índice](#indice)

___
<div id="parte6"></div>

# 6. O modelo final, o teste e o ponto de corte

Escolhido o `C`, o caminho até o fim tem três passos.

**Passo 1: ajustar o modelo final no treino inteiro.** A validação cruzada
serviu para escolher; o modelo que vamos usar é ajustado com todos os acórdãos
do treino.

**Passo 2: abrir o teste, uma vez.** A AUC no teste é a estimativa honesta do
desempenho em casos novos. Complete com a base certa.

In [ ]:
final = make_pipeline(cria_preparo(), LogisticRegression(C=0.3, max_iter=3000))
final.fit(X_treino, y_treino)

prob_teste = final.predict_proba(____)[:, 1]
print("AUC no teste:", round(roc_auc_score(y_teste, prob_teste), 3))

**Passo 3: escolher o ponto de corte.** Até aqui, o modelo só **ordena** os
acórdãos por probabilidade de divergência. Para usá-lo, alguém precisa decidir a
partir de que probabilidade um caso é **marcado**. Esse corte não sai da AUC: sai
da **necessidade de quem vai usar** o modelo, como no Projeto 06.

Imagine que a defensoria consegue acompanhar de perto **20% dos recursos**. O
corte deve marcar cerca de 20% dos casos, e a pergunta passa a ser: com esse
limite, quantas divergências o modelo encontra (a **sensibilidade**)?

O corte também se escolhe **sem abrir o teste**. Usamos as probabilidades de
validação do treino: o `cross_val_predict` devolve, para cada acórdão do treino,
a probabilidade dada pelo modelo da rodada em que ele estava fora do ajuste.
São previsões em "casos novos" sem tocar no teste.

In [ ]:
prob_validacao = cross_val_predict(final, X_treino, y_treino, cv=pastas, method="predict_proba")[:, 1]

linhas = []
for corte in [0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.10]:
    marcado = prob_validacao >= corte
    acertos = (marcado & (y_treino == 1)).sum()
    linhas.append({
        "corte": corte,
        "marcados": f"{marcado.mean():.0%}",                      # quanto da fila vai para a equipe
        "sensibilidade": f"{acertos / y_treino.sum():.0%}",       # das divergências, quantas são marcadas
        "precisao": f"{acertos / max(marcado.sum(), 1):.0%}",     # dos marcados, quantos divergem
    })
pd.DataFrame(linhas)

**Pergunta 6.**

a) Com capacidade para 20% dos recursos, qual corte você escolheria? Quanto das
divergências a equipe pegaria, pela validação?

b) A precisão fica baixa em qualquer corte (poucos dos marcados de fato
divergem). Isso torna o modelo inútil para a defensoria? Compare com acompanhar
20% dos recursos sorteados ao acaso.

c) Se a defensoria dissesse "não podemos perder nenhuma divergência", o que
aconteceria com o corte e com o trabalho da equipe?

Por fim, conferimos o corte escolhido no teste, que até aqui estava guardado.

In [ ]:
corte = 0.03
marcado = prob_teste >= corte
print("marcados no teste:", f"{marcado.mean():.0%}")
print("divergências encontradas:", (marcado & (y_teste == 1)).sum(), "de", y_teste.sum())

**Pergunta 7.** O resultado no teste ficou parecido com o da validação? Por que
os números podem variar bastante nesta base?

**Resumo da ordem das escolhas.** Na validação cruzada, a **AUC** escolhe o
modelo e o `C`, porque não depende do corte. Com o modelo final, o **corte** é
escolhido pela métrica que interessa a quem vai usar (aqui, a sensibilidade com
um limite de casos marcados), ainda com as probabilidades de validação. O teste
só confirma, uma vez.

[Volta ao Índice](#indice)